In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
# train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
# test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

In [3]:
from datasets import load_dataset

# Load data
dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)["train"]

# Create the combined_text column
dataset = dataset.map(
    lambda x: {
        "combined_text": x["prompt"] + " " + x["A"]
    }
)

# Get the character length of row 51
text = dataset[51]["combined_text"]

print(text)
print("Character length:", len(text))

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Determine the correct option: What is the reason behind the designation of Class L dwarfs, and what is their color and composition? among the listed options. Class L dwarfs are hotter than M stars and are designated L because L is the remaining letter alphabetically closest to M. They are bright blue in color and are brightest in ultraviolet. Their atmosphere is hot enough to allow metal hydrides and alkali metals to be prominent in their spectra. Some of these objects have masses large enough to support hydrogen fusion and are therefore stars, but most are of substellar mass and are therefore brown dwarfs.
Character length: 614


In [4]:
len(dataset[51]["combined_text"])

614

In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print(tokenizer.vocab_size)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

30522


In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print(tokenizer.sep_token)      # [SEP]
print(tokenizer.sep_token_id)   # 102

[SEP]
102


In [7]:
from datasets import load_dataset
from transformers import AutoTokenizer

# Load train.csv
dataset = load_dataset("csv", data_files={"train": "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"})

# Training split
train_dataset = dataset["train"]

# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Convert the prompt column to a Python list
prompts = list(train_dataset["prompt"])

# Tokenize
encodings = tokenizer(
    prompts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print(encodings["input_ids"].shape)

torch.Size([2000, 128])


In [8]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

# Load dataset
dataset = load_dataset(
    "csv",
    data_files={"train": "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"}
)["train"]

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

# Prompt from row 0
text = dataset[0]["prompt"]

# Tokenize with DEFAULT settings
inputs = tokenizer(text, return_tensors="pt")

# Forward pass
outputs = model(**inputs)

# Shape of last_hidden_state
print(outputs.last_hidden_state.shape)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 31, 768])


In [9]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
import torch

# Load dataset
dataset = load_dataset(
    "csv",
    data_files={"train": "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"}
)["train"]

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

# Put model in evaluation mode
model.eval()

# Get prompt from row 0
text = dataset[0]["prompt"]

# Tokenize (default settings)
inputs = tokenizer(text, return_tensors="pt")

# Forward pass
with torch.no_grad():
    outputs = model(**inputs)

# Extract the [CLS] embedding (first token)
cls_embedding = outputs.last_hidden_state[0, 0]

# Sum of the first 5 values
answer = cls_embedding[:5].sum().item()

print("First 5 values:", cls_embedding[:5])
print("Sum:", round(answer, 4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


First 5 values: tensor([-0.4677, -0.0754, -0.2019, -0.0071, -0.4480])
Sum: -1.2001


In [10]:
from transformers import AutoTokenizer, AutoModel
import torch

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

model.eval()

text = "Light-ion fusion is a technique."

# Tokenize
inputs = tokenizer(text, return_tensors="pt")

# Print tokens so you can locate "fusion"
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print(tokens)

# Forward pass
with torch.no_grad():
    outputs = model(**inputs)

# Last layer attention
last_layer_attention = outputs.attentions[-1]

# First attention head
head0 = last_layer_attention[0, 0]   # (seq_len, seq_len)

# Find the index of "fusion"
fusion_index = tokens.index("fusion")
print("Fusion index:", fusion_index)

# Attention from [CLS] (index 0) to fusion
attention_weight = head0[0, fusion_index].item()

print("Attention weight:", attention_weight)
print("Rounded:", round(attention_weight, 4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
Fusion index: 4
Attention weight: 0.10247287899255753
Rounded: 0.1025


In [11]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util

# Load dataset
dataset = load_dataset(
    "csv",
    data_files={"train": "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"}
)["train"]

# Load the Sentence Transformer model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Get prompt and Option B from row 0
prompt = dataset[0]["prompt"]
option_b = dataset[0]["B"]

# Generate embeddings
prompt_embedding = model.encode(prompt, convert_to_tensor=True)
option_b_embedding = model.encode(option_b, convert_to_tensor=True)

# Cosine similarity
similarity = util.cos_sim(prompt_embedding, option_b_embedding)

print("Similarity:", similarity)
print("Rounded:", round(similarity.item(), 4))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Similarity: tensor([[0.7658]])
Rounded: 0.7658


In [12]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm
import numpy as np

# -------------------------
# Load dataset
# -------------------------
dataset = load_dataset(
    "csv",
    data_files={"train": "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"}
)["train"]

# -------------------------
# MAP@3 function
# -------------------------
def map3(actuals, predictions):
    score = 0.0
    for actual, pred in zip(actuals, predictions):
        if actual in pred:
            score += 1.0 / (pred.index(actual) + 1)
    return score / len(actuals)

labels = ["A", "B", "C", "D", "E"]

###########################################################
# Pipeline 1 : TF-IDF
###########################################################

tfidf_predictions = []

for row in tqdm(dataset, desc="TF-IDF"):

    docs = [
        row["prompt"],
        row["A"],
        row["B"],
        row["C"],
        row["D"],
        row["E"],
    ]

    vectorizer = TfidfVectorizer(stop_words="english")
    X = vectorizer.fit_transform(docs)

    scores = cosine_similarity(X[0], X[1:]).flatten()

    ranking = np.argsort(scores)[::-1]

    tfidf_predictions.append([labels[i] for i in ranking[:3]])

###########################################################
# Pipeline 2 : MiniLM
###########################################################

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

minilm_predictions = []

for row in tqdm(dataset, desc="MiniLM"):

    prompt_embedding = model.encode(
        row["prompt"],
        convert_to_tensor=True
    )

    option_embeddings = model.encode(
        [row["A"], row["B"], row["C"], row["D"], row["E"]],
        convert_to_tensor=True
    )

    similarities = util.cos_sim(
        prompt_embedding,
        option_embeddings
    )[0].cpu().numpy()

    ranking = np.argsort(similarities)[::-1]

    minilm_predictions.append([labels[i] for i in ranking[:3]])

###########################################################
# Evaluation
###########################################################

answers = [row["answer"] for row in dataset]

minilm_map3 = map3(answers, minilm_predictions)

improved = sum(
    (
        answers[i] not in tfidf_predictions[i]
        and
        answers[i] in minilm_predictions[i]
    )
    for i in range(len(dataset))
)

print("MiniLM MAP@3 =", round(minilm_map3, 6))
print("Improved count =", improved)

TF-IDF:   0%|          | 0/2000 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MiniLM:   0%|          | 0/2000 [00:00<?, ?it/s]

MiniLM MAP@3 = 0.423083
Improved count = 488


In [13]:
from datasets import load_dataset
from transformers import pipeline

# Load dataset
dataset = load_dataset(
    "csv",
    data_files={"train": "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"}
)["train"]

# Initialize zero-shot classifier
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

# Prompt from row index 1
prompt = dataset[1]["prompt"]

# Candidate labels: Options A, B, and C
candidate_labels = [
    dataset[1]["A"],
    dataset[1]["B"],
    dataset[1]["C"]
]

# Run zero-shot classification
result = classifier(
    prompt,
    candidate_labels=candidate_labels
)

print(result)
print("Top label:", result["labels"][0])
print("Top score:", round(result["scores"][0], 4))

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion

In [14]:
from datasets import load_dataset
from transformers import pipeline

# Load dataset
dataset = load_dataset(
    "csv",
    data_files={"train": "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"}
)["train"]

# Zero-shot classifier
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

prompt = dataset[1]["prompt"]

candidate_labels = [
    dataset[1]["A"],
    dataset[1]["B"],
    dataset[1]["C"]
]

# Q10: Softmax (default)
result_softmax = classifier(
    prompt,
    candidate_labels=candidate_labels
)

# Q11: Independent Sigmoids
result_sigmoid = classifier(
    prompt,
    candidate_labels=candidate_labels,
    multi_label=True
)

softmax_sum = sum(result_softmax["scores"])
sigmoid_sum = sum(result_sigmoid["scores"])

difference = abs(softmax_sum - sigmoid_sum)

print("Softmax scores:", result_softmax["scores"])
print("Softmax sum:", softmax_sum)

print("Sigmoid scores:", result_sigmoid["scores"])
print("Sigmoid sum:", sigmoid_sum)

print("Absolute difference:", difference)
print("Rounded:", round(difference, 4))

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Softmax scores: [0.4574527442455292, 0.2750643789768219, 0.26748284697532654]
Softmax sum: 0.9999999701976776
Sigmoid scores: [0.00046927088988013566, 2.0635825421777554e-05, 1.970058110600803e-05]
Sigmoid sum: 0.0005096072964079212
Absolute difference: 0.9994903629012697
Rounded: 0.9995


In [15]:
from datasets import load_dataset
from transformers import pipeline

# Load dataset
dataset = load_dataset(
    "csv",
    data_files={"train": "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"}
)["train"]

# Load FLAN-T5 Small
generator = pipeline(
    "text-generation",
    model="google/flan-t5-small"
)

row = dataset[0]

input_text = (
    f"Question: {row['prompt']}. "
    f"Is the correct answer A: {row['A']} or B: {row['B']}? "
    f"Answer with just the letter A or B."
)

result = generator(
    input_text,
    max_new_tokens=5
)

print(result)
print("answer:")
print(result[0]["generated_text"])

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausa

[{'generated_text': "Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.. Is the correct answer A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time. or B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.? Answer with just the letter A or B."}]
answer:
Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.. Is the correct answer A: 